In [1]:
%%capture
!pip install unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# fourbit_models = [
#     "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
#     "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#     "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
#     "unsloth/llama-3-8b-Instruct-bnb-4bit",
#     "unsloth/llama-3-70b-bnb-4bit",
#     "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
#     "unsloth/Phi-3-medium-4k-instruct",
#     "unsloth/mistral-7b-bnb-4bit",
#     "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
# ] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-7B-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.1.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.191 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [4]:
saved_adapter_path = "final_math_adapter"
model.load_adapter(saved_adapter_path, adapter_name="default")

<All keys matched successfully>

In [5]:
# from datasets import load_dataset, Dataset
import os
from datasets import Dataset# Import os to check file existence

book_file_path = "cleaned_book_batch1.md"
paper_file_path = "cleaned_batch1.md" # Corrected path assuming this is intended
val_file_path = "cleaned_batch2.md"
val_lines_to_load = 10000

# --- Check if files exist ---
print(f"Checking existence of: {book_file_path}, {paper_file_path}, {val_file_path}")
missing_files = []
if not os.path.exists(book_file_path):
    missing_files.append(book_file_path)
if not os.path.exists(paper_file_path):
    missing_files.append(paper_file_path)
if not os.path.exists(val_file_path):
    missing_files.append(val_file_path)

if missing_files:
    print(f"Error: The following data files were not found:")
    for f in missing_files:
        print(f"- {f}")
    # Optional: List directories for debugging
    print("\nListing contents of /kaggle/input/ for context:")
    try:
        print(os.listdir("/kaggle/input/"))
        if os.path.exists(os.path.dirname(book_file_path)): print(f"Contents of {os.path.dirname(book_file_path)}: {os.listdir(os.path.dirname(book_file_path))}")
        if os.path.exists(os.path.dirname(paper_file_path)): print(f"Contents of {os.path.dirname(paper_file_path)}: {os.listdir(os.path.dirname(paper_file_path))}")
        if os.path.exists(os.path.dirname(val_file_path)): print(f"Contents of {os.path.dirname(val_file_path)}: {os.listdir(os.path.dirname(val_file_path))}")
    except Exception as e:
        print(f"Could not list input directory contents: {e}")
    raise FileNotFoundError(f"Required data files not found: {', '.join(missing_files)}")
else:
    print("All required data files found.")
# --- End Check ---


# --- Load and Combine Training Data ---
print("Loading and combining training data...")
book_text = ""
with open(book_file_path, 'r', encoding='utf-8') as f:
    book_text = f.read()

paper_text = ""
with open(paper_file_path, 'r', encoding='utf-8') as f:
    paper_text = f.read()

# Combine book first, then paper
combined_train_text = book_text + "\n" + paper_text # Add newline separator

# Create combined training dataset object
train_dataset_raw = Dataset.from_dict({"text": [combined_train_text]})
print("Raw Training Dataset created:")
print(train_dataset_raw)


# --- Load Limited Validation Data ---
print(f"Loading {val_lines_to_load} lines of validation data...")
val_lines = []
with open(val_file_path, 'r', encoding='utf-8') as f:
    val_lines = f.readlines()[:val_lines_to_load]
val_text = "".join(val_lines)

# Create validation dataset object
eval_dataset_raw = Dataset.from_dict({"text": [val_text]})
print("Raw Validation Dataset created:")
print(eval_dataset_raw)

Checking existence of: cleaned_book_batch1.md, cleaned_batch1.md, cleaned_batch2.md
All required data files found.
Loading and combining training data...
Raw Training Dataset created:
Dataset({
    features: ['text'],
    num_rows: 1
})
Loading 10000 lines of validation data...
Raw Validation Dataset created:
Dataset({
    features: ['text'],
    num_rows: 1
})


In [6]:
def tokenize_and_chunk(examples):
    tokenized_output = tokenizer(examples["text"], truncation=False, add_special_tokens=False) # Avoid adding BOS/EOS here if tokenizer does it automatically

    concatenated_examples = {k: sum(tokenized_output[k], []) for k in tokenized_output.keys()}
    total_length = len(concatenated_examples[list(tokenized_output.keys())[0]])

    if total_length >= max_seq_length:
        total_length = (total_length // max_seq_length) * max_seq_length
    else:
         # Handle case where total tokens < max_seq_length if needed, maybe pad later?
         # For CPT, dropping the remainder is common. If total < max_seq, result will be empty.
         print(f"Warning: Total token length ({total_length}) is less than max_seq_length ({max_seq_length}). No chunks generated.")
         return {"input_ids": [], "attention_mask": [], "labels": []} # Return empty


    result = {
        k: [t[i : i + max_seq_length] for i in range(0, total_length, max_seq_length)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

num_proc_tok = 1


# --- Tokenize Datasets ---
print(f"Tokenizing combined training dataset (max_seq_length = {max_seq_length})...")
tokenized_train_dataset = train_dataset_raw.map(
    tokenize_and_chunk,
    batched=True,
    num_proc=num_proc_tok,
    remove_columns=["text"],
)
print("Tokenized Training Dataset:")
print(tokenized_train_dataset)


print(f"Tokenizing validation dataset (max_seq_length = {max_seq_length})...")
tokenized_eval_dataset = eval_dataset_raw.map(
    tokenize_and_chunk,
    batched=True,
    num_proc=num_proc_tok,
    remove_columns=["text"],
)
print("Tokenized Validation Dataset:")
print(tokenized_eval_dataset)

# --- Validation Set Size Check ---
num_train_rows = len(tokenized_train_dataset)
num_eval_rows = len(tokenized_eval_dataset)
print(f"\nNumber of training sequences (rows): {num_train_rows}")
print(f"Number of validation sequences (rows): {num_eval_rows}")

if num_train_rows > 0 and num_eval_rows > 0 :
    eval_percentage = (num_eval_rows / (num_train_rows + num_eval_rows)) * 100
    print(f"Validation set size is approx. {eval_percentage:.2f}% of the total tokenized sequences.")
elif num_train_rows == 0:
     print("Warning: No training sequences generated after tokenization. Check data and max_seq_length.")
elif num_eval_rows == 0:
     print("Warning: No validation sequences generated after tokenization. Check data and max_seq_length.")

Tokenizing combined training dataset (max_seq_length = 2048)...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenized Training Dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 964
})
Tokenizing validation dataset (max_seq_length = 2048)...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenized Validation Dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 122
})

Number of training sequences (rows): 964
Number of validation sequences (rows): 122
Validation set size is approx. 11.23% of the total tokenized sequences.


In [8]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
eval_save_logging_steps = 20
trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_train_dataset,
    data_collator = data_collator,
    max_seq_length = max_seq_length,
    eval_dataset = tokenized_eval_dataset,

    args = UnslothTrainingArguments(
        # --- Batch Size ---
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,        # Effective batch size = 64

        # --- Training Duration: Exactly 1 Epoch ---
        # num_train_epochs = 1,                    # Train for exactly one full pass over the data
        max_steps = 60, # Ensure max_steps is not limiting if num_train_epochs is set

        # --- Learning Rate & Schedule ---
        learning_rate = 1e-5,                    # Keeping LR the same for this 1-epoch test
        embedding_learning_rate = 1e-6,          # Keeping LR the same for this 1-epoch test
        lr_scheduler_type = "cosine",            # Cosine schedule over 1 epoch
        warmup_steps = 5,                      # Warmup over 10% of the steps in 1 epoch (~1-2 steps)

        # --- Precision & Optimizer ---
        fp16 = False,
        bf16 = True,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,

        # --- Logging ---
        logging_strategy = "steps",              # Log based on steps
        logging_steps = 1, # Log frequently (every 2 steps)

        # --- Evaluation ---
        eval_strategy = "steps",           # Evaluate based on steps
        eval_steps = eval_save_logging_steps,    # Evaluate frequently (every 2 steps)

        # --- Saving ---
        save_strategy = "steps",                 # Save only at the end of the epoch
        # save_steps is ignored when save_strategy = "epoch"
        save_total_limit = 3,                    # Keep only the final epoch checkpoint

        # --- Misc ---
        seed = 3407,
        output_dir = "outputs_math_cpt_1epoch",  # New directory name for this run
        report_to = "none",
    ),
)

In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 964 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 1,170,735,104/7,000,000,000 (16.72% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,0.869800,0.913950
40,0.899900,0.912833
60,0.860800,0.912667


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [14]:
print("Training finished. Saving final adapter model...")

# Define the output directory in the Kaggle working directory
final_adapter_path = "final_math_adapter"

# Save the LoRA adapters
model.save_pretrained(final_adapter_path)

# Optionally, save the tokenizer as well (recommended)
tokenizer.save_pretrained(final_adapter_path)

print(f"Adapter model saved to: {final_adapter_path}")

Training finished. Saving final adapter model...
Adapter model saved to: final_math_adapter


In [2]:
print("\nReloading base model and applying the saved adapter for inference...")

from unsloth import FastLanguageModel
import torch
from peft import PeftModel 

base_model_name = "unsloth/Qwen2.5-Math-7B-bnb-4bit" 
final_adapter_path = "outputs_math_cpt_1epoch/checkpoint-60" 
dtype = None
load_in_4bit = True
max_seq_length = 2048

# --- Step 1: Load the Base Model ---
print(f"Loading base model: {base_model_name}")
model_inf, tokenizer_inf = FastLanguageModel.from_pretrained(
    model_name = base_model_name, 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Base model loaded.")

print(f"Applying adapter weights from: {final_adapter_path}")

model_inf = PeftModel.from_pretrained(model_inf, final_adapter_path)
print("Adapter weights applied.")

# --- Step 3: Prepare for Inference ---
model_inf.eval() # Set the model to evaluation mode

print("Model ready for inference with the fine-tuned adapter.")


Reloading base model and applying the saved adapter for inference...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model: unsloth/Qwen2.5-Math-7B-bnb-4bit
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.1.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.191 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Base model loaded.
Applying adapter weights from: outputs_math_cpt_1epoch/checkpoint-60
Adapter weights applied.
Model ready for inference with the fine-tuned adapter.


In [6]:
# Prompt that sets up the context for an equation
prompt = '''Explain the Function-Theoretic Center Problem in the context of complex dynamics, 
            specifically concerning the behavior of a conformal mapping 
                $z_{1} = f(z) = \lambda z + a_{2} z^{2} + \dots$ near the fixed point $z=0$.'''

inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=1024, # Reduced max tokens to see output sooner
        use_cache=True,
        pad_token_id=tokenizer_inf.eos_token_id,
        repetition_penalty=1.05
    )
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

Model Response:
 Explain the Function-Theoretic Center Problem in the context of complex dynamics, 
            specifically concerning the behavior of a conformal mapping 
                $z_{1} = f(z) = \lambda z + a_{2} z^{2} + \dots$ near the fixed point $z=0$. How does this relate to the stability of the fixed point and the existence of an invariant curve?
The function-theoretic center problem is a fundamental question in the study of complex dynamics, particularly concerning the behavior of a conformal mapping \( z_1 = f(z) = \lambda z + a_2 z^2 + \dots \) near a fixed point \( z = 0 \). Here, we assume that \( f(0) = 0 \) and \( f'(0) = \lambda \), where \( \lambda \) is a complex number. The problem is to determine whether there exists a local change of coordinates that conjugates \( f \) to its linear part \( z \mapsto \lambda z \).

### Step-by-Step Explanation

1. **Fixed Point and Linearization:**
   - A fixed point \( z = 0 \) of a function \( f \) is a point such that \( 

In [10]:
# Prompt that sets up the context for an equation
prompt = '''Derive the Jacobi integral, showing how it arises from the equations of motion in the rotating frame. Define the Jacobi constant $C$ and explain its role as a conserved quantity in this system.'''

inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=2048, # Reduced max tokens to see output sooner
        use_cache=True,
        pad_token_id=tokenizer_inf.eos_token_id,
        repetition_penalty=1.05
    )
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

Model Response:
 Derive the Jacobi integral, showing how it arises from the equations of motion in the rotating frame. Define the Jacobi constant $C$ and explain its role as a conserved quantity in this system. To derive the Jacobi integral, we start with the equations of motion for the three-body problem in the rotating frame. The equations of motion for the three-body problem in the inertial frame are given by:

\[
m_i \ddot{\mathbf{r}}_i = -\sum_{j \neq i} \frac{G m_i m_j (\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}
\]

where $\mathbf{r}_i$ is the position vector of the $i$-th body, $m_i$ is its mass, and $G$ is the gravitational constant. In the rotating frame, we introduce the relative positions $\mathbf{r}_{ij} = \mathbf{r}_j - \mathbf{r}_i$ and the relative velocities $\mathbf{v}_{ij} = \dot{\mathbf{r}}_j - \dot{\mathbf{r}}_i$. The equations of motion in the rotating frame become:

\[
m_i \ddot{\mathbf{r}}_i = -\sum_{j \neq i} \frac{G m_i m_j (\mathbf{r}_{ij})

In [9]:
# Prompt that sets up the context for an equation
prompt = r'''Describe how the equilibrium points, known as Libration points (or Lagrange points), are determined within the CRTBP framework, typically as critical points of the effective potential $\Phi$. Explain how the different types of Libration points (collinear $L_1, L_2, L_3$ and triangular $L_4, L_5$) arise mathematically from solving the relevant equations (e.g., $\partial \Phi / \partial \xi = 0, \partial \Phi / \partial \eta = 0$).'''
inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=1024, # Reduced max tokens to see output sooner
        use_cache=True,
        pad_token_id=tokenizer_inf.eos_token_id,
        repetition_penalty=1.05
    )
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

Model Response:
 Describe how the equilibrium points, known as Libration points (or Lagrange points), are determined within the CRTBP framework, typically as critical points of the effective potential $\Phi$. Explain how the different types of Libration points (collinear $L_1, L_2, L_3$ and triangular $L_4, L_5$) arise mathematically from solving the relevant equations (e.g., $\partial \Phi / \partial \xi = 0, \partial \Phi / \partial \eta = 0$). Discuss the stability of these points by analyzing the eigenvalues of the Hessian matrix of second derivatives of $\Phi$.

Assistant: To solve this problem, we need to follow several steps:

#### Step 1: Deriving the Equations of Motion

The equations of motion for the CRTBP can be derived using Newton's laws. The gravitational force between two masses $m_i$ and $m_j$ at positions $\mathbf{r}_i$ and $\mathbf{r}_j$ is given by:
$$
F_{ij} = -\frac{G m_i m_j}{|\mathbf{r}_i - \mathbf{r}_j|^3}(\mathbf{r}_i - \mathbf{r}_j)
$$
where $G$ is the gravit

In [ ]:
# Prompt that sets up the context for an equation
prompt = r'''Describe how the equilibrium points, known as Libration points (or Lagrange points), are determined within the CRTBP framework, typically as critical points of the effective potential $\Phi$. Explain how the different types of Libration points (collinear $L_1, L_2, L_3$ and triangular $L_4, L_5$) arise mathematically from solving the relevant equations. You aren't allowed to '''
inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=1024, # Reduced max tokens to see output sooner
        use_cache=True,
        pad_token_id=tokenizer_inf.eos_token_id,
        repetition_penalty=1.05
    )
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

In [3]:
# Prompt that sets up the context for an equation
prompt = r'''Explain the stability properties of the homographic solutions (relative equilibria) that arise from the 
            Lagrange equilateral triangle central configuration in the Newtonian N-body problem.'''
inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=1024, # Reduced max tokens to see output sooner
        use_cache=True,
        pad_token_id=tokenizer_inf.eos_token_id,
        repetition_penalty=1.05
    )
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

Model Response:
 Explain the stability properties of the homographic solutions (relative equilibria) that arise from the 
            Lagrange equilateral triangle central configuration in the Newtonian N-body problem. 

The first part of this question is answered by the following theorem, which is a special case of Theorem 1.2 in [10]. 

Theorem 3.1. Let $q_{1}, \ldots, q_{N}$ be the vertices of an equilateral triangle with side length $\ell$ and let $m_{1}=m_{2}=m_{3}=\frac{1}{\sqrt{3}}$. Then there exists a unique positive value of $\ell$ such that the configuration is a central configuration for the three-body problem. Moreover, the corresponding central configuration is a relative equilibrium solution of the three-body problem.

The second part of the question is answered by the following theorem, which is a special case of Theorem 1.4 in [10].

Theorem 3.2. Let $q_{1}, \ldots, q_{N}$ be the vertices of an equilateral triangle with side length $\ell$ and let $m_{1}=m_{2}=m_{3}=\fr